# Lab 3.3: Holiday Data Processing and Dataset Merging

**Name:** Zubair Moeen  
**Reg Number:** 22jzele0463  
**Lab:** Machine Learning Lab  
**Lab Supervisor:** Engr.Irshad Ullah  
**University:** UET Peshawar - Campus Nowshera  

## Lab Overview
This notebook prepares the time-series dataset for further machine learning work by adding holiday information. The code normalizes datetime values, loads processed holiday data, drops unnecessary columns, and merges both datasets.

## Learning Objectives
- Load the outlier-identified AEP dataset.
- Create a normalized date column from the Datetime column.
- Load and inspect processed holiday data.
- Merge holiday information with the main time-series dataset.

## Section 1: Dataset Loading and Date Preparation
This section loads the outlier-identified dataset and creates a separate Date column for merging with holiday records.


In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv(r'Z:\University\8th Semester\ML Lab\Lab 3\3_Outlier_Identified.csv',parse_dates=['Datetime'])
df.head()

,Datetime,AEP_MW
0,2004-10-01 01:00:00,12379.0
1,2004-10-01 02:00:00,11935.0
2,2004-10-01 03:00:00,11692.0
3,2004-10-01 04:00:00,11597.0
4,2004-10-01 05:00:00,11681.0


In [4]:
# df['Datetime'].dt.normalize() removes the time part from each timestamp and keeps only the date
df['Date'] = df['Datetime'].dt.normalize()

In [5]:
df.iloc[0:50]

,Datetime,AEP_MW,Date
0,2004-10-01 01:00:00,12379.0,2004-10-01
1,2004-10-01 02:00:00,11935.0,2004-10-01
2,2004-10-01 03:00:00,11692.0,2004-10-01
3,2004-10-01 04:00:00,11597.0,2004-10-01
4,2004-10-01 05:00:00,11681.0,2004-10-01
5,2004-10-01 06:00:00,12280.0,2004-10-01
6,2004-10-01 07:00:00,13692.0,2004-10-01
7,2004-10-01 08:00:00,14618.0,2004-10-01
8,2004-10-01 09:00:00,14903.0,2004-10-01
9,2004-10-01 10:00:00,15118.0,2004-10-01


In [6]:
len(df)

121296

In [7]:
holiday=pd.read_csv(r'Z:\University\8th Semester\ML Lab\Lab 3\processed_holiday.csv',parse_dates=['Date'])

In [8]:
holiday.info()

<class 'pandas.DataFrame'>
RangeIndex: 279 entries, 0 to 278
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Date     279 non-null    datetime64[us]
 1   Holiday  279 non-null    str           
 2   WeekDay  279 non-null    str           
 3   Month    279 non-null    int64         
 4   Day      279 non-null    int64         
 5   Year     279 non-null    int64         
dtypes: datetime64[us](1), int64(3), str(2)
memory usage: 13.2 KB


## Section 2: Holiday Dataset Inspection
The following cells load the processed holiday file, inspect its structure, and remove columns that are not required for merging.


In [9]:
holiday.head()

,Date,Holiday,WeekDay,Month,Day,Year
0,2004-01-01,New Year's Day,Thursday,1,1,2004
1,2004-01-19,"Martin Luther King, Jr. Day",Monday,1,19,2004
2,2004-02-14,Valentine’s Day,Saturday,2,14,2004
3,2004-02-16,Washington's Birthday,Monday,2,16,2004
4,2004-04-11,Eastern Easter,Sunday,4,11,2004


In [10]:
holiday.drop(['WeekDay','Month','Day','Year'],axis=1, inplace=True)

In [11]:
(len(holiday)*24)-(21*24) #in 2004 first 9 month is not include which includes 11 holidays.
                           #similarly in 2018 last 5 month not included which incluede 10 holidays.  21 total 

6192

In [12]:
mergedf = pd.merge(df, holiday, on= 'Date', how="left")
mergedf

,Datetime,AEP_MW,Date,Holiday
0,2004-10-01 01:00:00,12379.0,2004-10-01,NaN
1,2004-10-01 02:00:00,11935.0,2004-10-01,NaN
2,2004-10-01 03:00:00,11692.0,2004-10-01,NaN
3,2004-10-01 04:00:00,11597.0,2004-10-01,NaN
4,2004-10-01 05:00:00,11681.0,2004-10-01,NaN
...,...,...,...,...
121291,2018-08-02 20:00:00,17673.0,2018-08-02,NaN
121292,2018-08-02 21:00:00,17303.0,2018-08-02,NaN
121293,2018-08-02 22:00:00,17001.0,2018-08-02,NaN
121294,2018-08-02 23:00:00,15964.0,2018-08-02,NaN


In [13]:
#
mergedf['Holiday'].isna().sum()

np.int64(115104)

In [14]:
#121416
121296-mergedf['Holiday'].isna().sum()

np.int64(6192)

## .notnull()
 Creates a binary holiday feature where 1 means a holiday exists and 0 means no holiday.


Checks each row:

Returns True if the value is not null

Returns False if the value is null

## Section 3: Merging Main Data with Holiday Features
This section combines the AEP dataset with holiday information so that calendar effects can be used in later feature engineering.


In [15]:

mergedf['Holiday'] = mergedf['Holiday'].notnull().astype("int")

In [16]:
mergedf.head()

,Datetime,AEP_MW,Date,Holiday
0,2004-10-01 01:00:00,12379.0,2004-10-01,0
1,2004-10-01 02:00:00,11935.0,2004-10-01,0
2,2004-10-01 03:00:00,11692.0,2004-10-01,0
3,2004-10-01 04:00:00,11597.0,2004-10-01,0
4,2004-10-01 05:00:00,11681.0,2004-10-01,0


In [17]:
mergedf.to_csv(r'Z:\University\8th Semester\ML Lab\Lab 3\4_AEP_Introducing_holidays.csv',index=False)

In [18]:
mergedf[236:274]

,Datetime,AEP_MW,Date,Holiday
236,2004-10-10 21:00:00,14162.0,2004-10-10,0
237,2004-10-10 22:00:00,13770.0,2004-10-10,0
238,2004-10-10 23:00:00,13099.0,2004-10-10,0
239,2004-10-11 00:00:00,12346.0,2004-10-11,1
240,2004-10-11 01:00:00,11907.0,2004-10-11,1
241,2004-10-11 02:00:00,11614.0,2004-10-11,1
242,2004-10-11 03:00:00,11581.0,2004-10-11,1
243,2004-10-11 04:00:00,11514.0,2004-10-11,1
244,2004-10-11 05:00:00,11761.0,2004-10-11,1
245,2004-10-11 06:00:00,12558.0,2004-10-11,1


In [19]:
df.isnull().sum()

Datetime      0
AEP_MW      666
Date          0
dtype: int64

## Final Conclusion
In this lab, holiday information was added to the time-series dataset. This creates a richer dataset for future feature extraction and machine learning model development.
